# CS 229 - Homework 1: Alien CalcGPT

Submit **PDF** of completed notebook on Canvas.

**Maximum points**: 15

<div style="margin-bottom: 15px; padding: 15px; color: #31708f; background-color: #d9edf7; border: 1px solid #bce8f1; border-radius: 5px;">

<b><font size=+2>Enter your information below:</font></b><br/><br/>

  <b>(full) Name</b>: Vedant Deepak Borkute
  <br/>
  <b>Student ID Number</b>:  862552981
  <br/><br/>

<b>By submitting this notebook, I assert that the work below is my own work, Except where explicitly cited, none of the portions of this notebook are duplicated from anyone else's work.</b>
</div>

# Overview

In this assignment you will compare different methods for adapting a language model to understand math problems written in an **alien symbolic language**.

The alien language transforms standard arithmetic into unfamiliar syntax:

| Alien | Meaning | Example |
|-------|---------|--------|
| `flarn(a, b)` | a + b | `flarn(3, 5)` = 8 |
| `trok(a, b)` | a * b | `trok(4, 3)` = 12 |
| `snib(a, b)` | a - b | `snib(7, 2)` = 5 |
| `glorp(x)` | return x | `glorp(flarn(3, 5))` = 8 |

You will implement **five methods** and compare their effectiveness:
1. Base Prompting — minimal instruction
2. System Prompting — define the alien operators
3. Chain-of-Thought — step-by-step reasoning
4. In-Context Learning — provide solved examples
5. LoRA Fine-Tuning — adapt the model weights

> You are not being evaluated on finding the best prompt, but rather on correct implementation.

Complete all sections marked `TODO` and answer the analysis questions in Part 6.

## Setup
Run these cells to load the model and data. **Do not modify.**

In [16]:
# Install dependencies (run this on Colab; skip if already installed locally)
# !pip install -q transformers torch peft accelerate tqdm

import torch
import re
import matplotlib.pyplot as plt
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

In [17]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)
print(f"Loaded {MODEL_NAME}")

Using device: cpu


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded Qwen/Qwen2.5-0.5B-Instruct


## Load Data

Download `hw1_data.pt` from Canvas and place it in the same directory as this notebook.

The dataset contains arithmetic problems in alien notation, split into train (300) and test (150) sets at three difficulty levels:
- **Level 1**: single operation, e.g. `glorp(flarn(3, 5))`
- **Level 2**: one nested operation, e.g. `glorp(trok(4, flarn(2, 3)))`
- **Level 3**: two nested operations, e.g. `glorp(flarn(trok(2, 3), snib(10, 4)))`

In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
import os
os.listdir('drive/MyDrive')

['George4.pdf',
 'George3.pdf',
 'SECRET KEY TO UNIVERSE.pdf',
 'George2Lucy_HawkingSteph.pdf',
 'GEORGE 5.pdf',
 'My Docs',
 'Play Books Notes',
 'NilayCV.pdf',
 'Cosmos-Possible Worlds-National Geographic (2020).pdf',
 'Rio_de_Janeiro.pdf',
 'Mighty Raju Coming to Rio Song from Mighty Raju Rio Calling Movie.mp3',
 'PROOFS FOR RESUME VERIFICATION.gdoc',
 'Lost In Space _ Main Title Sequence (1).mp3',
 'Cosmos Main Title.mp3',
 'BBC Sherlock Theme Song.mp3',
 'Windows1.png',
 'Windows2.png',
 'WhatsApp Chat with Vedant.txt',
 'IMG-20191024-WA0019.jpg',
 'IMG-20191111-WA0000.jpg',
 'WhatsApp Chat with AAJI.txt',
 'Facebook.vcf',
 'WhatsApp Chat with Mum.txt',
 'Parth.vcf',
 'WhatsApp Chat with Bali Mavshi.txt',
 'WhatsApp Chat with Baba.txt',
 'M Baba.vcf',
 'AMIT.vcf',
 'Jijaji.vcf',
 'ubuntu.png',
 'ubuntu2.png',
 'Screenshot from 2021-02-19 02-26-18.png',
 'Screenshot from 2021-02-19 02-26-20.png',
 'Screenshot from 2021-02-19 02-28-45.png',
 'Screenshot from 2021-02-19 02-28-58.png'

In [20]:
data = torch.load('drive/MyDrive/HW 1 Alien CalcGPT.pt', weights_only=False)

In [21]:
# data = torch.load('hw1_data.pt', weights_only=False)
train_problems  = data['train_problems']
train_answers   = data['train_answers']
train_levels    = data['train_levels']
train_standard  = data['train_standard']   # standard math notation, e.g. "3 + 5"
test_problems   = data['test_problems']
test_answers    = data['test_answers']
test_levels     = data['test_levels']
test_standard   = data['test_standard']    # standard math notation, e.g. "3 + 5"
operators       = data['operators']

print(f"Train: {len(train_problems)} problems")
print(f"Test:  {len(test_problems)} problems")
print(f"\nAlien operators:")
for name, desc in operators.items():
    print(f"  {name}(...) = {desc}")
print(f"\nExample problems (one per difficulty level):")
for i in [0, 100, 200]:
    print(f"  Level {train_levels[i].item()}: {train_problems[i]}")
    print(f"  {'':>10}Standard math: {train_standard[i]} = {train_answers[i].item()}")

Train: 300 problems
Test:  150 problems

Alien operators:
  flarn(...) = addition (+)
  trok(...) = multiplication (*)
  snib(...) = subtraction (-)
  glorp(...) = return result (identity wrapper)

Example problems (one per difficulty level):
  Level 1: glorp(snib(8, 5))
            Standard math: 8 - 5 = 3
  Level 2: glorp(flarn(flarn(17, 10), 7))
            Standard math: (17 + 10) + 7 = 34
  Level 3: glorp(trok(trok(3, 4), flarn(1, 4)))
            Standard math: (3 * 4) * (1 + 4) = 60


## Helper Functions (Provided)

These functions handle model generation, answer extraction, and evaluation. **Do not modify** — your code will use them.

- `generate_response(messages, model, tokenizer)` — sends chat messages to the model and returns the text response
- `extract_answer(response)` — extracts the integer after `Final Answer:`
- `run_method(name, prompt_fn, problems, model, tokenizer)` — runs your prompt function on all test problems
- `print_results(...)` — prints accuracy overall and by difficulty level

In [22]:
def generate_response(messages, model, tokenizer, max_new_tokens=64):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


def extract_answer(response):
    """Extract the integer after 'Final Answer:' in a response string."""
    match = re.search(r"Final Answer:\s*(-?\d+)", response)
    return int(match.group(1)) if match else None


def run_method(name, prompt_fn, problems, model, tokenizer, max_new_tokens=64):
    """Run a prompting method on all problems; return (predictions, raw_outputs)."""
    predictions, raw_outputs = [], []
    for problem in tqdm(problems, desc=name):
        response = generate_response(prompt_fn(problem), model, tokenizer, max_new_tokens)
        predictions.append(extract_answer(response))
        raw_outputs.append(response)
    return predictions, raw_outputs


def print_results(name, predictions, answers, levels):
    """Print accuracy overall and by difficulty level. Returns overall accuracy."""
    correct = sum(1 for p, a in zip(predictions, answers) if p == a.item())
    total = len(answers)
    acc = correct / total
    print(f"\n{'=' * 55}")
    print(f"  {name}")
    print(f"  Overall: {correct}/{total} ({100*acc:.1f}%)")
    for level in [1, 2, 3]:
        idxs = [i for i in range(total) if levels[i].item() == level]
        lc = sum(1 for i in idxs if predictions[i] == answers[i].item())
        print(f"  Level {level}: {lc}/{len(idxs)} ({100*lc/len(idxs):.1f}%)")
    n_fail = sum(1 for p in predictions if p is None)
    if n_fail:
        print(f"  Format failures: {n_fail}")
    print(f"{'=' * 55}")
    return acc


# Dict to collect results from all methods for comparison plots
all_results = {}

In [23]:
# Quick demo: test the generation helper on a plain English math problem
demo = generate_response(
    [{"role": "user", "content": "What is 3 + 5? Respond with: Final Answer: <number>"}],
    model, tokenizer
)
print(f"Demo response: {demo}")
print(f"Extracted: {extract_answer(demo)}")

Demo response: Final Answer: 8
Extracted: 8


## Part 1: Base Prompting [1 point]

Implement `build_base_prompt(problem)` that returns a list of chat messages with **minimal instruction**.

**Requirements:**
- Do NOT include a system message
- Do NOT define the alien operators
- Return a list of dicts with `"role"` and `"content"` keys

This serves as your **baseline** — how well does the model do when it has no idea what the alien symbols mean?

In [24]:
def build_base_prompt(problem):
    """Base prompting: minimal instruction, no system prompt, no operator definitions.

    Args:
        problem: alien math expression string, e.g. "glorp(flarn(3, 5))"
    Returns:
        list of message dicts, e.g. [{"role": "user", "content": "..."}]
    """
    # TODO [1 point]: return a messages list with a single user message
    return [{'role': "user", 'content': f"What is {problem}? Respond with: Final Answer: <number>"}]
    pass

### Test your base prompt
The cell below runs your prompt on all test problems and prints accuracy.

**Expected runtime**: ~6 minutes on a T4 GPU. Base prompting is the slowest method because the model doesn't know when to stop and generates long outputs. If it's taking much longer, make sure you're using a GPU runtime (Runtime > Change runtime type > T4 GPU).

In [25]:
preds_base, outputs_base = run_method(
    "1. Base Prompting", build_base_prompt, test_problems, model, tokenizer, max_new_tokens=64
)
acc = print_results("1. Base Prompting", preds_base, test_answers, test_levels)
all_results["1. Base Prompting"] = {
    "predictions": preds_base, "raw_outputs": outputs_base, "accuracy": acc
}

1. Base Prompting: 100%|██████████| 150/150 [14:42<00:00,  5.88s/it]


  1. Base Prompting
  Overall: 8/150 (5.3%)
  Level 1: 6/50 (12.0%)
  Level 2: 1/50 (2.0%)
  Level 3: 1/50 (2.0%)


In [26]:
# Show a few example outputs
for i in range(3):
    status = "CORRECT" if preds_base[i] == test_answers[i].item() else "WRONG"
    print(f"Problem:  {test_problems[i]}")
    print(f"Expected: {test_answers[i].item()}  |  Got: {preds_base[i]}  [{status}]")
    print(f"Output:   {outputs_base[i][:200]}")
    print()

Problem:  glorp(flarn(9, 7))
Expected: 16  |  Got: 14  [WRONG]
Output:   Final Answer: 14

Problem:  glorp(snib(13, 10))
Expected: 3  |  Got: 13  [WRONG]
Output:   Final Answer: 13

Problem:  glorp(flarn(11, 16))
Expected: 27  |  Got: 11  [WRONG]
Output:   Final Answer: 11



## Part 2: System Prompting [2 points]

Implement `build_system_prompt(problem)` that uses a **system message** to define the alien operators.

**Requirements:**
- Include a system message that defines all 4 alien operators and their meanings
- Instruct the model to end its response with `Final Answer: <integer>`
- The user message should contain the problem to solve

In [27]:
def build_system_prompt(problem):
    """System prompting: define alien operators in the system message.

    Args:
        problem: alien math expression string
    Returns:
        list of message dicts with system and user messages
    """
    # TODO [2 points]: define a system prompt explaining the alien operators,
    # then return [system_message, user_message]
    system_message = "You are an expert mathematician who understands the following alien operators:\n"
    for name, desc in operators.items():
        system_message += f"- {name}(...) = {desc}\n"
    user_message = f"What is {problem}? Respond with: Final Answer: <number>"
    return [{'role': "system", 'content': system_message}, {'role': "user", 'content': user_message}]
    pass

### Test your system prompt

In [28]:
preds_system, outputs_system = run_method(
    "2. System Prompting", build_system_prompt, test_problems, model, tokenizer, max_new_tokens=64
)
acc = print_results("2. System Prompting", preds_system, test_answers, test_levels)
all_results["2. System Prompting"] = {
    "predictions": preds_system, "raw_outputs": outputs_system, "accuracy": acc
}

2. System Prompting: 100%|██████████| 150/150 [21:08<00:00,  8.46s/it]


  2. System Prompting
  Overall: 17/150 (11.3%)
  Level 1: 15/50 (30.0%)
  Level 2: 1/50 (2.0%)
  Level 3: 1/50 (2.0%)
  Format failures: 3


In [29]:
# Show a few example outputs
for i in range(3):
    status = "CORRECT" if preds_system[i] == test_answers[i].item() else "WRONG"
    print(f"Problem:  {test_problems[i]}")
    print(f"Expected: {test_answers[i].item()}  |  Got: {preds_system[i]}  [{status}]")
    print(f"Output:   {outputs_system[i][:200]}")
    print()

Problem:  glorp(flarn(9, 7))
Expected: 16  |  Got: 2  [WRONG]
Output:   Final Answer: 2

Problem:  glorp(snib(13, 10))
Expected: 3  |  Got: 3  [CORRECT]
Output:   Final Answer: 3

Problem:  glorp(flarn(11, 16))
Expected: 27  |  Got: 5  [WRONG]
Output:   Final Answer: 5.



## Part 3: Chain-of-Thought (CoT) [3 points]

Implement `build_cot_prompt(problem)` that asks the model to reason **step-by-step**.

Chain-of-thought prompting asks the model to show its intermediate reasoning before giving a final answer. See [Wei et al., 2022](https://arxiv.org/abs/2201.11903) for the original paper.

**Requirements:**
- Include the system prompt with operator definitions (same as Part 2)
- Additionally instruct the model to show step-by-step reasoning
- The model should translate each alien operator, evaluate inner expressions first, and show intermediate results
- Response format should be:
  ```
  Reasoning:
  <step-by-step work>
  Final Answer: <integer>
  ```
- Note: the test cell below uses `max_new_tokens=512` in `generate_response` to give the model enough room for step-by-step reasoning

In [30]:
def build_cot_prompt(problem):
    """Chain-of-Thought: system prompt with operator definitions + step-by-step reasoning.

    Args:
        problem: alien math expression string
    Returns:
        list of message dicts
    """
    # TODO [3 points]: build a prompt that includes operator definitions AND
    # requests step-by-step reasoning with a specific output format
    system_message = "You are an expert mathematician who understands the following alien operators:\n"
    for name, desc in operators.items():
        system_message += f"- {name}(...) = {desc}\n"
    system_message += "When you solve the problem, explain your reasoning step-by-step i.e., translate each alien operator, evaluate inner expressions first, and show intermediate results.\n"
    system_message += "- Response format should be:\n"
    system_message += "  ```\n"
    system_message += "  Reasoning:\n"
    system_message += "  <step-by-step work>\n"
    system_message += "  Final Answer: <integer>\n"
    system_message += "  ```\n"
    user_message = f"What is {problem}?"
    return [{'role': "system", 'content': system_message}, {'role': "user", 'content': user_message}]
    pass

### Test your CoT prompt

CoT generates longer outputs (step-by-step reasoning), so we test on a few specific problems with more token budget rather than the full test set. We pick one problem from each difficulty level. Examine the reasoning — can CoT solve hard problems that system prompting couldn't?

In [31]:
# CoT needs more tokens for reasoning, so we test on a few specific problems
# rather than the full test set. Pick one problem from each difficulty level.
cot_test_indices = [0, 50, 100]  # one Level 1, one Level 2, one Level 3

for idx in cot_test_indices:
    problem = test_problems[idx]
    expected = test_answers[idx].item()
    level = test_levels[idx].item()

    response = generate_response(build_cot_prompt(problem), model, tokenizer, max_new_tokens=512)
    pred = extract_answer(response)
    status = "CORRECT" if pred == expected else "WRONG"

    print(f"=== Level {level}: {problem} (expected: {expected}) [{status}] ===")
    print(response)
    print()

=== Level 1: glorp(flarn(9, 7)) (expected: 16) [WRONG] ===
Reasoning:
1. We need to apply the `trok(...)` operation first since it's a multiplication.
2. Then we add the result of that multiplication with the next operand.

Step-by-step calculation:
1. First, calculate the product of 9 and 7:
   - 9 * 7 = 63

2. Now, apply the `trok(...)` operation on this result:
   - trok(63) = 63 + 0 = 63

3. Finally, apply the `glorp(...)` operation on the result from step 2:
   - glorp(63) = 63 + 0 = 63

Final Answer: 63

=== Level 2: glorp(trok(10, flarn(11, 5))) (expected: 160) [WRONG] ===
To solve this problem, we need to follow these steps:

1. Evaluate the innermost expression first: trok(10, flarn(11, 5))
2. Use the result from step 1 as the input for the next operation.

Let's start with step 1:

```reasoning:
Step 1: trok(10, flarn(11, 5))
First, we perform the multiplication inside the parentheses:
flarn(11, 5) = 165
Then, we add 10 to get 115

Final answer: 115
```

Now let's move on to 

## Part 4: In-Context Learning (ICL) [3 points]

Implement `get_icl_examples(...)` to select solved examples from the **training set**, and `build_icl_prompt(problem)` to include them in the prompt.

**Requirements:**
- Select 3 examples from the training set — 1 per difficulty level (NOT from the test set!)
- Include examples as alternating user/assistant turns in the conversation
- Include the system prompt with operator definitions
- Hint: with a small model, fewer examples often work better — too many examples can overwhelm the model's limited context

In [35]:
def get_icl_examples(train_problems, train_answers, train_levels, n_per_level=1):
    """Select solved examples from the training set.

    Args:
        train_problems: list of alien expression strings
        train_answers: tensor of integer answers
        train_levels: tensor of difficulty levels (1, 2, 3)
        n_per_level: how many examples to pick per level
    Returns:
        list of (problem_string, answer_int) tuples
    """
    # TODO: select examples from the training data
    res = []
    for difficulty in [1, 2, 3]:
        idxs = [i for i in range(len(train_problems)) if train_levels[i].item() == difficulty]
        selected = idxs[:n_per_level] 
        examples = [(train_problems[i], train_answers[i].item()) for i in selected]
        print(f"Selected {len(examples)} examples for difficulty {difficulty}")
        for prob, ans in examples:
            res.append((prob, ans))
    return res


def build_icl_prompt(problem):
    """ICL: system prompt + solved examples as multi-turn user/assistant pairs.

    Args:
        problem: alien math expression string
    Returns:
        list of message dicts
    """
    # TODO: build messages list with system prompt, example turns, and the problem
    system_message = "You are an expert mathematician who understands the following alien operators:\n"
    user_message = f"What is {problem}?"
    for name, desc in operators.items():
        system_message += f"- {name}(...) = {desc}\n"
    messages = [{"role": "system", "content": system_message}]

    for ex_prob, ex_ans in icl_examples:
        messages.append({"role": "user", "content": f"What is {ex_prob}?"})
        messages.append({"role": "assistant", "content": f"Final Answer: {ex_ans}"})
    user_message = f"What is {problem}? Respond with: Final Answer: <number>"
    messages.append({"role": "user", "content": user_message})
    return messages
    pass


# TODO: call get_icl_examples to create the icl_examples variable
icl_examples = get_icl_examples(train_problems, train_answers, train_levels, n_per_level=1)

Selected 1 examples for difficulty 1
Selected 1 examples for difficulty 2
Selected 1 examples for difficulty 3


In [36]:
# Debug: inspect the full prompt to verify it looks correct
debug_msgs = build_icl_prompt(test_problems[0])
print("=== Full ICL prompt for first test problem ===")
for msg in debug_msgs:
    print(f"[{msg['role']}]: {msg['content']}")

=== Full ICL prompt for first test problem ===
[system]: You are an expert mathematician who understands the following alien operators:
- flarn(...) = addition (+)
- trok(...) = multiplication (*)
- snib(...) = subtraction (-)
- glorp(...) = return result (identity wrapper)

[user]: What is glorp(snib(8, 5))?
[assistant]: Final Answer: 3
[user]: What is glorp(flarn(flarn(17, 10), 7))?
[assistant]: Final Answer: 34
[user]: What is glorp(trok(trok(3, 4), flarn(1, 4)))?
[assistant]: Final Answer: 60
[user]: What is glorp(flarn(9, 7))? Respond with: Final Answer: <number>


### Test your ICL prompt

In [ ]:
preds_icl, outputs_icl = run_method(
    "4. In-Context Learning", build_icl_prompt, test_problems, model, tokenizer, max_new_tokens=64
)
acc = print_results("4. In-Context Learning", preds_icl, test_answers, test_levels)
all_results["4. In-Context Learning"] = {
    "predictions": preds_icl, "raw_outputs": outputs_icl, "accuracy": acc
}

4. In-Context Learning:  27%|██▋       | 40/150 [10:31<29:06, 15.87s/it]

In [ ]:
# Show a few example outputs
for i in range(3):
    status = "CORRECT" if preds_icl[i] == test_answers[i].item() else "WRONG"
    print(f"Problem:  {test_problems[i]}")
    print(f"Expected: {test_answers[i].item()}  |  Got: {preds_icl[i]}  [{status}]")
    print(f"Output:   {outputs_icl[i][:200]}")
    print()

## Part 5: LoRA Fine-Tuning [5 points]

Fine-tune the model using [LoRA (Low-Rank Adaptation)](https://arxiv.org/abs/2106.09685) so it learns the alien language from training examples.

After fine-tuning, the model should solve problems **without** needing operator definitions in the prompt — because the knowledge is now in the weights.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import DataLoader, TensorDataset

### Part 5a: Configure LoRA [1 point]

Create a `LoraConfig` for causal language modeling. You need to choose:
- `r` — the rank of the low-rank matrices (try 4–16)
- `lora_alpha` — scaling factor (commonly 2×r)
- `target_modules` — which attention layers to adapt (e.g., `["q_proj", "v_proj"]`)

In [ ]:
# TODO 5a [1 point]: Create a LoraConfig
lora_config = LoraConfig(r = 10, lora_alpha=32, target_modules=["q_proj", "v_proj"])

### Part 5b: Prepare Data and Train [4 points]

The training data preparation function is provided below. It formats each (problem, answer) pair as a chat conversation and tokenizes it, masking the prompt tokens (set to `-100`) so the model only learns to predict the answer.

Your job: write the training loop.
- Use `AdamW` optimizer with learning rate `2e-4`
- Train for `3` epochs with batch size `4`
- Print average loss each epoch

**Computing the loss**: Run the forward pass *without* `labels=` and compute the loss manually using `torch.nn.functional.cross_entropy`. For causal LM training, you need to shift the logits and labels so they align (the model predicts the *next* token at each position). Also, use ignore_index=-100 in cross_entropy to skip masked positions. 


In [ ]:
# Provided: prepare training data (do not modify)
def prepare_lora_data(problems, answers, tokenizer, max_length=128):
    """Tokenize (problem, answer) pairs as chat conversations for LoRA training."""
    all_input_ids, all_labels, all_attn = [], [], []
    for problem, answer in zip(problems, answers):
        full_msgs = [
            {"role": "user", "content": f"Solve: {problem}"},
            {"role": "assistant", "content": f"Final Answer: {answer}"}
        ]
        prompt_msgs = [{"role": "user", "content": f"Solve: {problem}"}]
        full_text = tokenizer.apply_chat_template(full_msgs, tokenize=False)
        prompt_text = tokenizer.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)
        prompt_len = len(tokenizer(prompt_text)["input_ids"])
        enc = tokenizer(full_text, max_length=max_length, truncation=True,
                         padding="max_length", return_tensors="pt")
        ids = enc["input_ids"].squeeze(0)
        attn = enc["attention_mask"].squeeze(0)
        labels = ids.clone()
        labels[:prompt_len] = -100
        labels[attn == 0] = -100
        all_input_ids.append(ids)
        all_labels.append(labels)
        all_attn.append(attn)
    return torch.stack(all_input_ids), torch.stack(all_labels), torch.stack(all_attn)

train_ids, train_labels_lora, train_attn = prepare_lora_data(
    train_problems, [a.item() for a in train_answers], tokenizer
)
print(f"Training data shape: {train_ids.shape}")

In [ ]:
# Inspect one training example to verify formatting
i = 0
full_text = tokenizer.decode(train_ids[i][train_attn[i] == 1])
label_tokens = train_labels_lora[i][train_labels_lora[i] != -100]
label_text = tokenizer.decode(label_tokens)
print("=== Full training string ===")
print(full_text)
print("\n=== Model learns to predict (unmasked labels) ===")
print(label_text)

In [ ]:
# Reload a fresh model and apply LoRA (do not modify this cell)
lora_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
)
lora_model = get_peft_model(lora_model, lora_config)
lora_model.print_trainable_parameters()

In [ ]:
# TODO 5b [4 points]: Write the training loop
# - Create a DataLoader from train_ids, train_labels_lora, train_attn (batch_size=4, shuffle=True)
# - Use AdamW optimizer with lr=2e-4
# - Train for 3 epochs
# - Each batch: forward pass with (input_ids, attention_mask, labels), backward, step
# - Print average loss per epoch
# - Use cross entropy loss (don't use outputs.loss)

n_epochs = 3
learning_rate = 2e-4

# TODO: create DataLoader, optimizer, and training loop

### Evaluate LoRA

In [ ]:
def build_lora_prompt(problem):
    """LoRA: minimal prompt — the fine-tuned model has learned the alien language."""
    return [{"role": "user", "content": f"Solve: {problem}"}]

lora_model.eval()
preds_lora, outputs_lora = run_method(
    "5. LoRA", build_lora_prompt, test_problems, lora_model, tokenizer
)
acc = print_results("5. LoRA Fine-Tuning", preds_lora, test_answers, test_levels)
all_results["5. LoRA"] = {
    "predictions": preds_lora, "raw_outputs": outputs_lora, "accuracy": acc
}

In [ ]:
# Show a few example outputs
for i in range(3):
    status = "CORRECT" if preds_lora[i] == test_answers[i].item() else "WRONG"
    print(f"Problem:  {test_problems[i]}")
    print(f"Expected: {test_answers[i].item()}  |  Got: {preds_lora[i]}  [{status}]")
    print(f"Output:   {outputs_lora[i][:200]}")
    print()

## Part 6: Analysis [1 point]

Review the plots and failure examples below, then briefly discuss: which methods worked best, what kinds of failures you observed, and why you think some methods outperformed others.

In [ ]:
methods = list(all_results.keys())
accuracies = [all_results[m]["accuracy"] * 100 for m in methods]

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6', '#f39c12']
bars = ax.bar(methods, accuracies, color=colors)
ax.set_ylabel("Accuracy (%)")
ax.set_title("Overall Accuracy by Method")
ax.set_ylim(0, 100)
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f"{acc:.1f}%", ha='center', fontsize=10)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
methods = list(all_results.keys())
fig, ax = plt.subplots(figsize=(12, 5))
x = range(len(methods))
width = 0.25
level_colors = ['#3498db', '#e67e22', '#e74c3c']

for li, level in enumerate([1, 2, 3]):
    level_accs = []
    for m in methods:
        preds = all_results[m]["predictions"]
        idxs = [j for j in range(len(test_levels)) if test_levels[j].item() == level]
        lc = sum(1 for j in idxs if preds[j] == test_answers[j].item())
        level_accs.append(100 * lc / len(idxs))
    ax.bar([xi + li * width for xi in x], level_accs, width,
           label=f"Level {level}", color=level_colors[li])

ax.set_ylabel("Accuracy (%)")
ax.set_title("Accuracy by Method and Difficulty Level")
ax.set_xticks([xi + width for xi in x])
ax.set_xticklabels(methods, rotation=15, ha='right')
ax.legend()
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

In [ ]:
def show_failures(method_name, n=3):
    preds = all_results[method_name]["predictions"]
    outputs = all_results[method_name]["raw_outputs"]
    print(f"\n--- Sample failures: {method_name} ---")
    count = 0
    for i in range(len(preds)):
        if preds[i] != test_answers[i].item():
            print(f"  Problem:  {test_problems[i]}")
            print(f"  Standard: {test_standard[i]} = {test_answers[i].item()}")
            print(f"  Got:      {preds[i]}")
            print(f"  Output:   {outputs[i][:200]}")
            print()
            count += 1
            if count >= n:
                break
    if count == 0:
        print("  No failures!")

for m in all_results:
    show_failures(m)

### Your Analysis

*TODO [1 point]: Write a short paragraph (4–6 sentences) discussing your results.*



## Extra Credit

Extra credit is submitted separately on Canvas using the **HW 1 (EC)** assignment. You must submit a PDF (made with LaTeX) outlining the problem, your approach, results (including tables or figures), and a discussion of results. Also submit your code (.py or .ipynb). Extra credit is graded on quality, generally 1–5 points, though could be higher for novel research directions. If you have a direction, feel free to pitch it!

### EC1: Robustness & Variance
Run the same prompts multiple times with `do_sample=True` and `temperature > 0`. Measure the variance in answers and consistency across the different methods. How stable are the results? Which methods are most/least reliable?

For a deeper study, analyze using the methods in the ["hot mess" paper](https://arxiv.org/abs/2601.23045).

### EC2: Confidence Calibration
Examine the confidence calibration of all methods. The goal is to get `p(final answer | prompt, output reasoning)` for each prediction. You can compute this by running a forward pass with `model()` on the generated output and extracting the logits for the answer tokens. Use `torch.softmax` to convert logits to probabilities.

If all probabilities are low, try normalizing over a restricted answer set: compute `p(final answer = i)` for `i = 0, 1, ..., 500` and renormalize. This is a common technique for structured outputs.

Use 10 confidence bins to build a reliability diagram and compute the Expected Calibration Error (ECE) as described in [Guo et al., 2017](https://arxiv.org/abs/1706.04599).

### EC3: Adversarial Alien Obfuscation
Compare different ways of obfuscating the math. For instance, if we use something semantic like "smash" instead of "glorp" to represent addition, is it easier or harder? Can you design strings of characters that reliably prevent learning at all?

See [AutoPrompt (Shin et al., EMNLP 2020)](https://aclanthology.org/2020.emnlp-main.346.pdf) for methods to automatically search for such prompts.

### EC4: Discrete Diffusion
Repeat the homework using [LLaDA](https://arxiv.org/abs/2502.09992), a discrete diffusion language model, and report any differences in performance. LLaDA allows sampling using fewer steps — does this dramatically affect any of the strategies (maybe all of them)?

### EC5: Reinforcement Learning with Verifiable Rewards
Instead of supervised fine-tuning (LoRA), try using reinforcement learning to train the model. Since our alien math problems have verifiable correct answers, we can use the answer as a reward signal. Implement Group Relative Policy Optimization (GRPO) as described in the [DeepSeekMath paper (Shao et al., 2024)](https://arxiv.org/abs/2402.03300). Compare the RL-trained model's accuracy against the LoRA supervised approach.

## Submission

Export this notebook to PDF and submit on Canvas.
```python
!jupyter nbconvert --to pdf --output=yourname_hw1.pdf hw1.ipynb
```